In [ ]:
# LightGBM Boosting + Optuna (Kaggle Irrigation Need)

This notebook is **LightGBM-only** and follows the same style as prior class notebooks:

1. Build multiple LightGBM configurations (at least 3 different hyperparameter sets).
2. Compare cross-validated performance to see which settings matter.
3. Tune LightGBM with Optuna.
4. Train the best LightGBM model on full data and create a Kaggle submission.

In [3]:
%pip install lightgbm optuna -q


Note: you may need to restart the kernel to use updated packages.


In [4]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, balanced_accuracy_score, accuracy_score

import lightgbm as lgb
import optuna

In [5]:
# Load data
train = pd.read_csv("../Kaggle/train.csv")
test = pd.read_csv("../Kaggle/test.csv")

TARGET_COL = "Irrigation_Need"
ID_COL = "id"

feature_cols = [c for c in train.columns if c not in [ID_COL, TARGET_COL]]

X_train_raw = train[feature_cols].copy()
X_test_raw = test[feature_cols].copy()
y_raw = train[TARGET_COL].copy()

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Target distribution:")
print(y_raw.value_counts(normalize=True).round(4))

Train shape: (630000, 21)
Test shape: (270000, 20)
Target distribution:
Irrigation_Need
Low       0.5872
Medium    0.3795
High      0.0333
Name: proportion, dtype: float64


In [6]:
# Encode target
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

print("Label mapping:")
for cls, enc in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)):
    print(f"  {cls} -> {enc}")

# Keep preprocessing identical for train and test
all_features = pd.concat([X_train_raw, X_test_raw], axis=0)
all_features = pd.get_dummies(all_features, drop_first=False)

X = all_features.iloc[: len(X_train_raw)].copy()
X_test = all_features.iloc[len(X_train_raw):].copy()

print("Encoded train feature shape:", X.shape)
print("Encoded test feature shape:", X_test.shape)

Label mapping:
  High -> 0
  Low -> 1
  Medium -> 2
Encoded train feature shape: (630000, 43)
Encoded test feature shape: (270000, 43)


In [ ]:
# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

base_params = {
    "objective": "multiclass",
    "num_class": len(label_encoder.classes_),
    "boosting_type": "gbdt",
    "random_state": 42,
    "verbosity": -1,
    "n_jobs": -1,
}

manual_param_sets = {
    "set_1_baseline": {
        "n_estimators": 300,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "max_depth": -1,
        "min_child_samples": 20,
        "subsample": 1.0,
        "subsample_freq": 0,
        "colsample_bytree": 1.0,
        "reg_alpha": 0.0,
        "reg_lambda": 0.0,
    },
    "set_2_regularized": {
        "n_estimators": 700,
        "learning_rate": 0.03,
        "num_leaves": 63,
        "max_depth": 12,
        "min_child_samples": 80,
        "subsample": 0.8,
        "subsample_freq": 1,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.2,
        "reg_lambda": 0.6,
    },
    "set_3_aggressive": {
        "n_estimators": 1000,
        "learning_rate": 0.02,
        "num_leaves": 127,
        "max_depth": 14,
        "min_child_samples": 30,
        "subsample": 0.9,
        "subsample_freq": 1,
        "colsample_bytree": 0.9,
        "reg_alpha": 0.05,
        "reg_lambda": 0.2,
    },
}

manual_results = []
for name, p in manual_param_sets.items():
    params = base_params.copy()
    params.update(p)

    model = lgb.LGBMClassifier(**params)
    scores = cross_val_score(model, X, y, cv=cv, scoring="balanced_accuracy", n_jobs=-1)

    manual_results.append(
        {
            "config": name,
            "cv_balanced_accuracy_mean": scores.mean(),
            "cv_balanced_accuracy_std": scores.std(),
        }
    )

manual_results_df = pd.DataFrame(manual_results).sort_values("cv_balanced_accuracy_mean", ascending=False)
manual_results_df

In [ ]:
def objective(trial):
    params = base_params.copy()
    params.update(
        {
            "n_estimators": trial.suggest_int("n_estimators", 300, 1400),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 16, 256),
            "max_depth": trial.suggest_int("max_depth", 4, 16),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 200),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "subsample_freq": trial.suggest_int("subsample_freq", 1, 7),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        }
    )

    model = lgb.LGBMClassifier(**params)
    scores = cross_val_score(model, X, y, cv=cv, scoring="balanced_accuracy", n_jobs=-1)
    return scores.mean()

In [ ]:
# Run Optuna tuning
N_TRIALS = 35

study = optuna.create_study(direction="maximize", study_name="lgbm_irrigation_optuna")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_manual = manual_results_df.iloc[0]["cv_balanced_accuracy_mean"]
print("Best manual-set CV Balanced Accuracy:", round(float(best_manual), 5))
print("Best Optuna CV Balanced Accuracy:", round(study.best_value, 5))
print("Improvement over best manual set:", round(study.best_value - float(best_manual), 5))

print("\nBest Optuna params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
# Fit tuned model on full training set
best_params = study.best_params.copy()
best_params.update(
    {
        "objective": "multiclass",
        "num_class": len(label_encoder.classes_),
        "random_state": 42,
        "verbosity": -1,
        "n_jobs": -1,
    }
)

best_model = lgb.LGBMClassifier(**best_params)
best_model.fit(X, y)

# Optional quick self-check on train data (not leaderboard metric)
train_pred = best_model.predict(X)
print("Train accuracy:", round(accuracy_score(y, train_pred), 5))
print("Train balanced accuracy:", round(balanced_accuracy_score(y, train_pred), 5))
print("\nTrain classification report:")
print(classification_report(y, train_pred, target_names=label_encoder.classes_))

In [ ]:
# Predict test and create submission
test_pred_encoded = best_model.predict(X_test)
test_pred_labels = label_encoder.inverse_transform(test_pred_encoded.astype(int))

submission = pd.DataFrame(
    {
        ID_COL: test[ID_COL],
        TARGET_COL: test_pred_labels,
    }
)

submission_path = "../Kaggle/submission_lgbm_optuna.csv"
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
submission.head()

## Next improvements (LightGBM-only)

- Increase `N_TRIALS` (e.g., 100+) if you have time.
- Use `optuna.importance.get_param_importances(study)` to see which LightGBM hyperparameters matter most.
- Keep CV scoring as `balanced_accuracy` to match the Kaggle leaderboard metric.
- Add early stopping via fold-level train/validation loops to reduce overfitting and runtime.
- Try DART boosting (`boosting_type='dart'`) as an additional LightGBM variant.